<a href="https://colab.research.google.com/github/WaitUps/CourseraExercises-AI-Agents-and-Agentic-AI-Course/blob/main/ProgrammaticPrompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1: Agentic AI Exercise - Gemini Adaptation for Response Generation

This section focuses on the foundational aspects of integrating with Google's Generative AI models using the Python SDK. We'll cover API key management, model discovery, and implementing a versatile helper function for generating model responses.

### 1.1 Key Concepts

**Key features and concepts demonstrated:**

*   **API Key Management:** Securely configuring your Google API Key from Colab secrets.
*   **Model Discovery:** Listing available Generative AI models.
*   **Flexible Response Generation:** Implementing a `generate_response` helper function designed to:
    *   Interact seamlessly with various Gemini models.
    *   Handle system instructions (passing them directly or integrating into chat history based on model compatibility).
    *   Format messages correctly for the Gemini API.

The `generate_response` function is a component, built for adaptability and robust handling of model interactions, including detailed examples of how to interpret model outputs.

In [1]:
from google.colab import userdata # Imports the userdata module to access secrets stored in Google Colab
import google.genai as genai # Imports the Google Generative AI SDK

# Retrieve the API key from Colab's secrets manager.
# The key should be named 'GOOGLE_API_KEY' in the secrets tab.
api_key = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=api_key) # Initialize the client with the API key

In [2]:
#import google.genai as genai

for m in client.models.list():
  # Check if the model supports the 'generateContent' method.
  # This is crucial for models that are intended for text generation and conversational AI.
  if "generateContent" in m.supported_actions:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [3]:
from typing import List, Dict # Used for type hinting to improve code readability and maintainability
import google.genai as genai # Imports the Google Generative AI SDK
import json # Import json for pretty printing

# The 'client' object is expected to be globally available from the API key setup cell (KEYrzG2vB8Ip).
# It provides the direct interface for interacting with the Gemini API service.

def generate_response(messages: List[Dict]) -> tuple[str, str]:
    """
    Calls the Gemini LLM to generate a response based on a list of messages using the direct client interface.
    It handles system instructions by incorporating them as the first 'user' message in the content list.
    Returns a tuple: (generated_text: str, detailed_markdown_log: str).
    """

    #model_to_use = 'gemini-flash-latest'
    #model_to_use = 'gemma-3-1b-it' #Gemma models are generally smaller and may have different quota availability
    model_to_use = 'gemini-flash-lite-latest'
    generation_config_dict = {'max_output_tokens':4096}


    detailed_markdown_log_parts = []
    detailed_markdown_log_parts.append("#### `generate_response` Call Details\n\n")

    detailed_markdown_log_parts.append("**Input Messages:**\n")
    detailed_markdown_log_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n\n")

    formatted_messages_for_client = []
    for msg in messages:
        # CRITICAL ADAPTATION FOR `client.models.generate_content`:
        # This specific method of the Gemini API does not have a distinct 'system_instruction' parameter.
        # Instead, system-level instructions or persona settings must be provided as the first 'user' turn.
        # Therefore, if the input message has a 'system' role, we convert it to a 'user' role
        # for compatibility with the API call. This effectively sets the context for the model.
        actual_role = msg['role']
        if actual_role == 'system':
            # Convert 'system' role to 'user' role for API compatibility.
            # This ensures that system instructions are correctly interpreted by the model
            # as part of the conversation history, setting the overall context or persona.
            actual_role = 'user'
        formatted_messages_for_client.append({'role': actual_role, 'parts': [{'text': msg['content']}]})

    detailed_markdown_log_parts.append("**Formatted Messages for `client.models.generate_content`:**\n")
    detailed_markdown_log_parts.append(f"```json\n{json.dumps(formatted_messages_for_client, indent=2)}\n```\n\n")


    detailed_markdown_log_parts.append(f"**Model Used:** `{model_to_use}`\n\n")
    detailed_markdown_log_parts.append("**Generation Configuration:**\n")
    detailed_markdown_log_parts.append(f"```json\n{json.dumps(generation_config_dict, indent=2)}\n```\n\n")

    # Content Generation using the direct client interface:
    # Instead of instantiating a GenerativeModel, we call `generate_content` directly on `client.models`.
    # The 'model' argument specifies which Gemini model to use.
    # 'contents' is the list of formatted messages forming the conversation history.
    # `generation_config` controls generation parameters like `max_output_tokens`.
    response = client.models.generate_content(
        model=model_to_use, # Specify the model directly
        contents=formatted_messages_for_client,
        config=generation_config_dict
    )

    detailed_markdown_log_parts.append("**LLM Raw Response (text content):**\n")
    detailed_markdown_log_parts.append(f"\n{response.text}\n\n\n")

    final_markdown_output = "".join(detailed_markdown_log_parts)

    # Return Value:
    # The generated text content from the model's response is extracted and returned
    # along with the detailed markdown log.
    return response.text, final_markdown_output

In [7]:
# For the `client.models.generate_content` API endpoint, a dedicated 'system' role is not used.
# Instead, system instructions (like defining the AI's persona) are typically provided
# as the first 'user' message in the conversation history. This sets the context for the model.
# Therefore, the generate_response function will convert all 'system' roles into 'user' roles.

messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response, markdown_output = generate_response(messages)

In [8]:
print(markdown_output)

#### `generate_response` Call Details

**Input Messages:**
```json
[
  {
    "role": "system",
    "content": "You are an expert software engineer that prefers functional programming."
  },
  {
    "role": "user",
    "content": "Write a function to swap the keys and values in a dictionary."
  }
]
```

**Formatted Messages for `client.models.generate_content`:**
```json
[
  {
    "role": "user",
    "parts": [
      {
        "text": "You are an expert software engineer that prefers functional programming."
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Write a function to swap the keys and values in a dictionary."
      }
    ]
  }
]
```

**Model Used:** `gemini-flash-lite-latest`

**Generation Configuration:**
```json
{
  "max_output_tokens": 4096
}
```

**LLM Raw Response (text content):**

In functional programming, we prioritize immutability and declarative transformations. To swap keys and values in a dictionary, we treat the dictionary as a co

## Sample of Detailed Markdown Log [final_markdown_output]  using `gemini-flash-lite` (Agentic AI Exercise: Section 1):

#### `generate_response` Call Details

**Input Messages:**
```json
[
  {
    "role": "system",
    "content": "You are an expert software engineer that prefers functional programming."
  },
  {
    "role": "user",
    "content": "Write a function to swap the keys and values in a dictionary."
  }
]
```

**Formatted Messages for `client.models.generate_content`:**
```json
[
  {
    "role": "user",
    "parts": [
      {
        "text": "You are an expert software engineer that prefers functional programming."
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Write a function to swap the keys and values in a dictionary."
      }
    ]
  }
]
```

**Model Used:** `gemini-flash-lite-latest`

**Generation Configuration:**
```json
{
  "max_output_tokens": 4096
}
```

**LLM Raw Response (text content):**

To swap keys and values in a dictionary using a functional approach, we leverage **dictionary comprehensions**. This is the idiomatic Python equivalent of mapping over a collection.

### The Implementation

```python
from typing import Mapping, TypeVar, Hashable

K = TypeVar('K', bound=Hashable)
V = TypeVar('V', bound=Hashable)

def invert_dict(data: Mapping[K, V]) -> dict[V, K]:
    """
    Returns a new dictionary with keys and values swapped.
    
    Note: If the input dictionary contains duplicate values,
    the last occurrence will overwrite previous ones in the result.
    """
    return {value: key for key, value in data.items()}
```

### Why this fits the Functional Paradigm:

1.  **Immutability:** The function does not mutate the original dictionary. Instead, it computes and returns a new dictionary, adhering to the principle of pure functions.
2.  **Declarative Syntax:** Rather than imperatively initializing an empty dictionary and manually looping with `.append()` or assignment statements, the dictionary comprehension describes *what* the output should be rather than *how* to build it step-by-step.
3.  **Conciseness:** It treats the transformation as a mapping operation, which is central to functional programming.

### Handling Collisions (Functional Pattern)
If your input dictionary has duplicate values, the standard comprehension will "lose" data (the last key wins). If you want to preserve all keys in a functional way, you can group them using `itertools`:

```python
from collections import defaultdict
from itertools import groupby

def invert_dict_multivalue(data: Mapping[K, V]) -> dict[V, list[K]]:
    """
    Inverts the dictionary, grouping original keys into a list
    if the values are duplicated.
    """
    # Sort items by value (required for groupby)
    sorted_items = sorted(data.items(), key=lambda item: item[1])
    
    return {
        val: [k for k, v in group]
        for val, group in groupby(sorted_items, key=lambda item: item[1])
    }
```

This version uses `groupby` to ensure no data is discarded, maintaining the integrity of the data transformation while remaining expressive and side-effect free.


## Sample of Generated Response [response.text] `using gemini-2.5-flash` (Agentic AI Exercise: Section 1):

In functional programming, we treat data as immutable and focus on transformations. To swap keys and values in a dictionary, we want to transform a collection of pairs $(k, v)$ into a collection of pairs $(v, k)$.

Here are a few ways to achieve this in Python, ranging from the idiomatic "Pythonic" functional style to a more rigorous approach that handles potential data loss.

### 1. The Declarative Approach (Dictionary Comprehension)
This is the most common way to perform a swap. It is declarative because it describes the relationship between the old and new data structures.

```python
from typing import Dict, TypeVar

K = TypeVar('K')
V = TypeVar('V')

def swap_dict(d: Dict[K, V]) -> Dict[V, K]:
    """Returns a new dictionary with keys and values swapped."""
    return {v: k for k, v in d.items()}
```

### 2. The Higher-Order Function Approach
If we want to avoid explicit loops entirely, we can use `map` and `reversed`. Since `d.items()` returns a sequence of `(key, value)` tuples, we can reverse each tuple to get `(value, key)`.

```python
def swap_dict_functional(d: Dict[K, V]) -> Dict[V, K]:
    # reversed() on a tuple (k, v) returns an iterator yielding v then k
    # dict() can consume an iterable of iterables (pairs)
    return dict(map(reversed, d.items())) # type: ignore
```

### 3. Handling Collisions (The "Fold" Approach)
In a standard swap, if two keys have the same value, one will be overwritten (last-one-in wins). A true functional engineer considers edge cases: **What if the mapping is not injective?**

To avoid losing data, we can use a "fold" (known as `reduce` in Python) to group keys into a list for each value.

```python
from functools import reduce
from typing import List

def swap_and_group(d: Dict[K, V]) -> Dict[V, List[K]]:
    """
    Swaps keys and values, grouping keys into lists to prevent data loss.
    This is a classic 'fold' pattern.
    """
    def accumulator(acc: Dict[V, List[K]], item: tuple[K, V]) -> Dict[V, List[K]]:
        key, value = item
        # In a purely functional language, we'd return a new dict here
        # To be performant in Python, we update the accumulator
        acc.setdefault(value, []).append(key)
        return acc

    return reduce(accumulator, d.items(), {})

# Example:
# input: {'a': 1, 'b': 2, 'c': 1}
# output: {1: ['a', 'c'], 2: ['b']}
```

### Which one should you use?
1.  **Use Example 1** for 99% of use cases. It is readable, efficient, and idiomatic.
2.  **Use Example 2** if you are strictly adhering to a "point-free" or piping style common in languages like Haskell or F#.
3.  **Use Example 3** if your dictionary values are not unique and you cannot afford to lose data during the transformation.

**Note on Purity:** In all these examples, the original dictionary `d` remains unchanged, satisfying the core FP principle of immutability.



# Section 2: Building a Quasi-Agent for Python Function Generation

This section implements a 'quasi-agent' that demonstrates how to maintain context across multiple prompts with an LLM to build a Python function step-by-step. It asks the user for a function description, then uses sequential calls to the `generate_response` function (our LLM wrapper) to:

1.  Generate the basic Python function.
2.  Add comprehensive documentation to the function.
3.  Generate `unittest` test cases for the function.

Key improvements:
- The `develop_custom_function` leverages a growing message history to iteratively refine the generated Python function, documentation, and test cases.
- This allows for multi-turn interactions where subsequent LLM calls build upon previous responses, improving the coherence and completeness of the generated artifacts.
- Facilitates a more robust and 'agentic' development workflow by extending the agent's memory.

This exercise highlights managing information flow and handling LLM outputs, even when they might be inconsistent. We will leverage the `generate_response` function we defined earlier, which is configured to use `gemini-flash-lite-latest` and handles system instructions and message formatting.

In [9]:
from typing import List, Dict
import sys
import os

def extract_code_block(response: str) -> str:
   """Extracts the first Python code block from a string response."""

   if '```' not in response:
      # If no code block markers, assume the entire response is code or not what we expect
      # For robustness, we might want to return an empty string or raise an error here
      return response.strip()

   # Split by ``` and take the content between the first two markers
   parts = response.split('```')
   if len(parts) < 3:
      # Not enough parts to contain a code block
      return response.strip()

   code_block = parts[1].strip()

   # Remove 'python' or 'py' if it's at the start of the code block
   if code_block.startswith("python"): # Covers 'python' and 'python\n'
      code_block = code_block[6:]
   elif code_block.startswith("py"): # Covers 'py' and 'py\n'
      code_block = code_block[2:]

   return code_block.strip()

In [10]:
from typing import List, Dict
import sys
import os
import json # Import json for pretty printing

def develop_custom_function():
   # Get user input for function description
   print("\nWhat kind of Python function would you like to create?")
   print("Example: 'A function that calculates the factorial of a number'")
   print("Your description: ", end='')
   function_description = input().strip()

   markdown_output_parts = [] # List to collect markdown strings

   # Initialize conversation messages.
   # For the `client.models.generate_content` API endpoint, a dedicated 'system' role is not used.
   # Instead, system instructions (like defining the AI's persona) are typically provided
   # as the first 'user' message in the conversation history. This sets the context for the model.
   messages = [
        {"role": "user", "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."}
   ]

   # --- STEP 1: Generating Initial Function ---
   markdown_output_parts.append("\n## STEP 1: Generating Initial Function\n")
   # First prompt - Basic function
   user_prompt_1 = f"Write a basic Python function that {function_description}."

   markdown_output_parts.append("### Prompt for Step 1:\n")
   markdown_output_parts.append(f"```\n{user_prompt_1}\n```\n")

   markdown_output_parts.append("### Messages (Memory) before Step 1:\n")
   markdown_output_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n")

   messages.append({"role": "user", "content": user_prompt_1})
   llm_raw_response_1, detailed_markdown_from_gen_response_1 = generate_response(messages)
   markdown_output_parts.append("#### LLM Response Generation Log (Step 1 - Initial Function)\n")
   markdown_output_parts.append(detailed_markdown_from_gen_response_1)

   initial_function_code = extract_code_block(llm_raw_response_1)
   markdown_function_code = f"```python\n{initial_function_code}\n```"

   markdown_output_parts.append("#### Extracted Function Code (Step 1)\n")
   markdown_output_parts.append(markdown_function_code + "\n") # Add newline after code block

   # Add LLM's response to conversation history. Use the raw extracted code, not the markdown-wrapped version.
   messages.append({"role": "model", "content": initial_function_code})

   markdown_output_parts.append("### Messages (Memory) after Step 1:\n")
   markdown_output_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n")

   # --- STEP 2: Adding Documentation ---
   markdown_output_parts.append("\n## STEP 2: Adding Documentation\n")
   # Second prompt - Add documentation
   user_prompt_2 = "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."

   markdown_output_parts.append("### Prompt for Step 2:\n")
   markdown_output_parts.append(f"```\n{user_prompt_2}\n```\n")

   markdown_output_parts.append("### Messages (Memory) before Step 2:\n")
   markdown_output_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n")

   messages.append({"role": "user", "content": user_prompt_2})
   llm_raw_response_2, detailed_markdown_from_gen_response_2 = generate_response(messages)
   markdown_output_parts.append("#### LLM Response Generation Log (Step 2 - Documented Function)\n")
   markdown_output_parts.append(detailed_markdown_from_gen_response_2)

   documented_function_code = extract_code_block(llm_raw_response_2)
   markdown_doc_function_code = f"```python\n{documented_function_code}\n```"

   markdown_output_parts.append("#### Extracted Documented Function Code (Step 2)\n")
   markdown_output_parts.append(markdown_doc_function_code + "\n") # Add newline after code block

   # Add documented function response to conversation history. Use the raw extracted code.
   messages.append({"role": "model", "content": documented_function_code})

   markdown_output_parts.append("### Messages (Memory) after Step 2:\n")
   markdown_output_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n")

   # --- STEP 3: Adding Test Cases ---
   markdown_output_parts.append("\n## STEP 3: Adding Test Cases\n")
   # Third prompt - Add test cases
   user_prompt_3 = "Finally, add unittest test cases for this function. Tests should cover basic functionality, edge cases, and various input scenarios. Output the test cases code in a separate ```python code block```."

   markdown_output_parts.append("### Prompt for Step 3:\n")
   markdown_output_parts.append(f"```\n{user_prompt_3}\n```\n")

   markdown_output_parts.append("### Messages (Memory) before Step 3:\n")
   markdown_output_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n")

   messages.append({"role": "user", "content": user_prompt_3})
   llm_raw_response_3, detailed_markdown_from_gen_response_3 = generate_response(messages)
   markdown_output_parts.append("#### LLM Response Generation Log (Step 3 - Test Cases)\n")
   markdown_output_parts.append(detailed_markdown_from_gen_response_3)

   test_cases_code = extract_code_block(llm_raw_response_3)
   markdown_test_cases_code = f"```python\n{test_cases_code}\n```"

   markdown_output_parts.append("#### Extracted Test Cases Code (Step 3)\n")
   markdown_output_parts.append(markdown_test_cases_code + "\n") # Add newline after code block

   # Add model response to messages history for completeness of 'memory after step 3' log. Use the raw extracted code.
   messages.append({"role": "model", "content": test_cases_code})

   markdown_output_parts.append("### Messages (Memory) after Step 3:\n")
   markdown_output_parts.append(f"```json\n{json.dumps(messages, indent=2)}\n```\n")

   # Generate a filename from the function description
   filename_base = function_description.lower().replace('a function that ', '').replace('a function to ', '').strip()
   filename_base = ''.join(c for c in filename_base if c.isalnum() or c.isspace())
   filename_base = filename_base.replace(' ', '_')
   filename = f"{filename_base[:30] if len(filename_base) > 30 else filename_base}.py"

   # Ensure the filename is not empty or just an extension
   if not filename_base:
       filename = "generated_function.py"

   # Save final version to a Python file
   full_code = documented_function_code + '\n\n' + test_cases_code
   with open(filename, 'w') as f:
      f.write(full_code)

   markdown_output_parts.append(f"\n## Final code has been saved to: {filename}\n")
   markdown_output_parts.append("\n---\n") # Add a horizontal rule for visual separation
   markdown_output_parts.append("### Quasi-Agent Development Process Complete")

   final_markdown_output = "".join(markdown_output_parts)

   return documented_function_code, test_cases_code, filename, final_markdown_output

### Run the Quasi-Agent

Execute the cell below to start the quasi-agent. You will be prompted to enter a description for the Python function you want to create. The agent will then proceed through the steps of generating the function, adding documentation, and creating test cases, printing its progress along the way.

In [11]:
# Call the quasi-agent to develop a custom function
function_code, tests, filename, markdown_output = develop_custom_function()


What kind of Python function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: Converts a decimal integer into its binary, octal, and hexadecimal string representations.


In [12]:
print(markdown_output)


## STEP 1: Generating Initial Function
### Prompt for Step 1:
```
Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations..
```
### Messages (Memory) before Step 1:
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  }
]
```
#### LLM Response Generation Log (Step 1 - Initial Function)
#### `generate_response` Call Details

**Input Messages:**
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  }
]
```

**Formatted Messages for `client.models.generate_content`:**
```json
[
  {
    "role": "user",
    "parts": [
      {
        "text":

## Sample of Detailed Markdown Log [final_markdown_output]  using `gemini-flash-lite` (A Quasi-Agent: Section 2):


## STEP 1: Generating Initial Function
### Prompt for Step 1:
```
Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations..
```
### Messages (Memory) before Step 1:
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  }
]
```
#### LLM Response Generation Log (Step 1 - Initial Function)
#### `generate_response` Call Details

**Input Messages:**
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  }
]
```

**Formatted Messages for `client.models.generate_content`:**
```json
[
  {
    "role": "user",
    "parts": [
      {
        "text": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
      }
    ]
  }
]
```

**Model Used:** `gemini-flash-lite-latest`

**Generation Configuration:**
```json
{
  "max_output_tokens": 4096
}
```

**LLM Raw Response (text content):**

To convert a decimal integer into binary, octal, and hexadecimal strings, Python provides built-in functions: `bin()`, `oct()`, and `hex()`.

Here is a function that returns these representations in a dictionary format:

```python
def convert_number_systems(n):
    """
    Converts a decimal integer into binary, octal, and hexadecimal strings.
    Note: Python's built-in functions include prefixes (0b, 0o, 0x).
    """
    return {
        "decimal": n,
        "binary": bin(n),
        "octal": oct(n),
        "hexadecimal": hex(n)
    }

# Example usage:
result = convert_number_systems(255)
print(result)
```

### Explanation:
*   **`bin(n)`**: Returns the binary string (e.g., `'0b11111111'`).
*   **`oct(n)`**: Returns the octal string (e.g., `'0o377'`).
*   **`hex(n)`**: Returns the hexadecimal string (e.g., `'0xff'`).

**Note:** If you prefer the output **without** the Python prefixes (`0b`, `0o`, `0x`), you can use string slicing `[2:]` or the `format()` function:

```python
def convert_number_systems_no_prefix(n):
    return {
        "binary": format(n, 'b'),
        "octal": format(n, 'o'),
        "hexadecimal": format(n, 'x')
    }
```


#### Extracted Function Code (Step 1)
```python
def convert_number_systems(n):
    """
    Converts a decimal integer into binary, octal, and hexadecimal strings.
    Note: Python's built-in functions include prefixes (0b, 0o, 0x).
    """
    return {
        "decimal": n,
        "binary": bin(n),
        "octal": oct(n),
        "hexadecimal": hex(n)
    }

# Example usage:
result = convert_number_systems(255)
print(result)
```
### Messages (Memory) after Step 1:
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
  }
]
```

## STEP 2: Adding Documentation
### Prompt for Step 2:
```
Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```.
```
### Messages (Memory) before Step 2:
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
  }
]
```
#### LLM Response Generation Log (Step 2 - Documented Function)
#### `generate_response` Call Details

**Input Messages:**
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
  },
  {
    "role": "user",
    "content": "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
  }
]
```

**Formatted Messages for `client.models.generate_content`:**
```json
[
  {
    "role": "user",
    "parts": [
      {
        "text": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
      }
    ]
  },
  {
    "role": "model",
    "parts": [
      {
        "text": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
      }
    ]
  }
]
```

**Model Used:** `gemini-flash-lite-latest`

**Generation Configuration:**
```json
{
  "max_output_tokens": 4096
}
```

**LLM Raw Response (text content):**

```python
def convert_number_systems(n):
    """
    Converts a decimal integer into its binary, octal, and hexadecimal string representations.

    This function utilizes Python's built-in conversion functions. Note that the
    returned strings include the standard Python prefixes: '0b' for binary,
    '0o' for octal, and '0x' for hexadecimal.

    Parameters:
    n (int): The decimal integer to be converted.

    Returns:
    dict: A dictionary containing the original integer and its string
          representations with the following keys:
          - 'decimal': The original integer.
          - 'binary': The binary string representation.
          - 'octal': The octal string representation.
          - 'hexadecimal': The hexadecimal string representation.

    Example:
    >>> convert_number_systems(255)
    {'decimal': 255, 'binary': '0b11111111', 'octal': '0o377', 'hexadecimal': '0xff'}

    Edge Cases:
    - If n is 0, returns {'decimal': 0, 'binary': '0b0', 'octal': '0o0', 'hexadecimal': '0x0'}.
    - If n is negative, the prefixes are preceded by a minus sign
      (e.g., -0b101).
    - If input is not an integer, a TypeError will be raised by the
      built-in conversion functions.
    """
    return {
        "decimal": n,
        "binary": bin(n),
        "octal": oct(n),
        "hexadecimal": hex(n)
    }
```


#### Extracted Documented Function Code (Step 2)
```python
def convert_number_systems(n):
    """
    Converts a decimal integer into its binary, octal, and hexadecimal string representations.

    This function utilizes Python's built-in conversion functions. Note that the
    returned strings include the standard Python prefixes: '0b' for binary,
    '0o' for octal, and '0x' for hexadecimal.

    Parameters:
    n (int): The decimal integer to be converted.

    Returns:
    dict: A dictionary containing the original integer and its string
          representations with the following keys:
          - 'decimal': The original integer.
          - 'binary': The binary string representation.
          - 'octal': The octal string representation.
          - 'hexadecimal': The hexadecimal string representation.

    Example:
    >>> convert_number_systems(255)
    {'decimal': 255, 'binary': '0b11111111', 'octal': '0o377', 'hexadecimal': '0xff'}

    Edge Cases:
    - If n is 0, returns {'decimal': 0, 'binary': '0b0', 'octal': '0o0', 'hexadecimal': '0x0'}.
    - If n is negative, the prefixes are preceded by a minus sign
      (e.g., -0b101).
    - If input is not an integer, a TypeError will be raised by the
      built-in conversion functions.
    """
    return {
        "decimal": n,
        "binary": bin(n),
        "octal": oct(n),
        "hexadecimal": hex(n)
    }
```
### Messages (Memory) after Step 2:
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
  },
  {
    "role": "user",
    "content": "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into its binary, octal, and hexadecimal string representations.\n\n    This function utilizes Python's built-in conversion functions. Note that the \n    returned strings include the standard Python prefixes: '0b' for binary, \n    '0o' for octal, and '0x' for hexadecimal.\n\n    Parameters:\n    n (int): The decimal integer to be converted.\n\n    Returns:\n    dict: A dictionary containing the original integer and its string \n          representations with the following keys:\n          - 'decimal': The original integer.\n          - 'binary': The binary string representation.\n          - 'octal': The octal string representation.\n          - 'hexadecimal': The hexadecimal string representation.\n\n    Example:\n    >>> convert_number_systems(255)\n    {'decimal': 255, 'binary': '0b11111111', 'octal': '0o377', 'hexadecimal': '0xff'}\n\n    Edge Cases:\n    - If n is 0, returns {'decimal': 0, 'binary': '0b0', 'octal': '0o0', 'hexadecimal': '0x0'}.\n    - If n is negative, the prefixes are preceded by a minus sign \n      (e.g., -0b101).\n    - If input is not an integer, a TypeError will be raised by the \n      built-in conversion functions.\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }"
  }
]
```

## STEP 3: Adding Test Cases
### Prompt for Step 3:
```
Finally, add unittest test cases for this function. Tests should cover basic functionality, edge cases, and various input scenarios. Output the test cases code in a separate ```python code block```.
```
### Messages (Memory) before Step 3:
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
  },
  {
    "role": "user",
    "content": "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into its binary, octal, and hexadecimal string representations.\n\n    This function utilizes Python's built-in conversion functions. Note that the \n    returned strings include the standard Python prefixes: '0b' for binary, \n    '0o' for octal, and '0x' for hexadecimal.\n\n    Parameters:\n    n (int): The decimal integer to be converted.\n\n    Returns:\n    dict: A dictionary containing the original integer and its string \n          representations with the following keys:\n          - 'decimal': The original integer.\n          - 'binary': The binary string representation.\n          - 'octal': The octal string representation.\n          - 'hexadecimal': The hexadecimal string representation.\n\n    Example:\n    >>> convert_number_systems(255)\n    {'decimal': 255, 'binary': '0b11111111', 'octal': '0o377', 'hexadecimal': '0xff'}\n\n    Edge Cases:\n    - If n is 0, returns {'decimal': 0, 'binary': '0b0', 'octal': '0o0', 'hexadecimal': '0x0'}.\n    - If n is negative, the prefixes are preceded by a minus sign \n      (e.g., -0b101).\n    - If input is not an integer, a TypeError will be raised by the \n      built-in conversion functions.\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }"
  }
]
```
#### LLM Response Generation Log (Step 3 - Test Cases)
#### `generate_response` Call Details

**Input Messages:**
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
  },
  {
    "role": "user",
    "content": "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into its binary, octal, and hexadecimal string representations.\n\n    This function utilizes Python's built-in conversion functions. Note that the \n    returned strings include the standard Python prefixes: '0b' for binary, \n    '0o' for octal, and '0x' for hexadecimal.\n\n    Parameters:\n    n (int): The decimal integer to be converted.\n\n    Returns:\n    dict: A dictionary containing the original integer and its string \n          representations with the following keys:\n          - 'decimal': The original integer.\n          - 'binary': The binary string representation.\n          - 'octal': The octal string representation.\n          - 'hexadecimal': The hexadecimal string representation.\n\n    Example:\n    >>> convert_number_systems(255)\n    {'decimal': 255, 'binary': '0b11111111', 'octal': '0o377', 'hexadecimal': '0xff'}\n\n    Edge Cases:\n    - If n is 0, returns {'decimal': 0, 'binary': '0b0', 'octal': '0o0', 'hexadecimal': '0x0'}.\n    - If n is negative, the prefixes are preceded by a minus sign \n      (e.g., -0b101).\n    - If input is not an integer, a TypeError will be raised by the \n      built-in conversion functions.\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }"
  },
  {
    "role": "user",
    "content": "Finally, add unittest test cases for this function. Tests should cover basic functionality, edge cases, and various input scenarios. Output the test cases code in a separate ```python code block```."
  }
]
```

**Formatted Messages for `client.models.generate_content`:**
```json
[
  {
    "role": "user",
    "parts": [
      {
        "text": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
      }
    ]
  },
  {
    "role": "model",
    "parts": [
      {
        "text": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
      }
    ]
  },
  {
    "role": "model",
    "parts": [
      {
        "text": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into its binary, octal, and hexadecimal string representations.\n\n    This function utilizes Python's built-in conversion functions. Note that the \n    returned strings include the standard Python prefixes: '0b' for binary, \n    '0o' for octal, and '0x' for hexadecimal.\n\n    Parameters:\n    n (int): The decimal integer to be converted.\n\n    Returns:\n    dict: A dictionary containing the original integer and its string \n          representations with the following keys:\n          - 'decimal': The original integer.\n          - 'binary': The binary string representation.\n          - 'octal': The octal string representation.\n          - 'hexadecimal': The hexadecimal string representation.\n\n    Example:\n    >>> convert_number_systems(255)\n    {'decimal': 255, 'binary': '0b11111111', 'octal': '0o377', 'hexadecimal': '0xff'}\n\n    Edge Cases:\n    - If n is 0, returns {'decimal': 0, 'binary': '0b0', 'octal': '0o0', 'hexadecimal': '0x0'}.\n    - If n is negative, the prefixes are preceded by a minus sign \n      (e.g., -0b101).\n    - If input is not an integer, a TypeError will be raised by the \n      built-in conversion functions.\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }"
      }
    ]
  },
  {
    "role": "user",
    "parts": [
      {
        "text": "Finally, add unittest test cases for this function. Tests should cover basic functionality, edge cases, and various input scenarios. Output the test cases code in a separate ```python code block```."
      }
    ]
  }
]
```

**Model Used:** `gemini-flash-lite-latest`

**Generation Configuration:**
```json
{
  "max_output_tokens": 4096
}
```

**LLM Raw Response (text content):**

Here are the unit tests for the `convert_number_systems` function, covering standard integers, zero, negative numbers, and error handling.

```python
import unittest

class TestConvertNumberSystems(unittest.TestCase):

    def test_positive_integer(self):
        result = convert_number_systems(10)
        self.assertEqual(result["binary"], "0b1010")
        self.assertEqual(result["octal"], "0o12")
        self.assertEqual(result["hexadecimal"], "0xa")

    def test_zero(self):
        result = convert_number_systems(0)
        self.assertEqual(result["binary"], "0b0")
        self.assertEqual(result["octal"], "0o0")
        self.assertEqual(result["hexadecimal"], "0x0")

    def test_negative_integer(self):
        result = convert_number_systems(-10)
        self.assertEqual(result["binary"], "-0b1010")
        self.assertEqual(result["octal"], "-0o12")
        self.assertEqual(result["hexadecimal"], "-0xa")

    def test_large_integer(self):
        result = convert_number_systems(1024)
        self.assertEqual(result["binary"], "0b10000000000")
        self.assertEqual(result["hexadecimal"], "0x400")

    def test_invalid_input(self):
        with self.assertRaises(TypeError):
            convert_number_systems("string")
        with self.assertRaises(TypeError):
            convert_number_systems(None)

if __name__ == '__main__':
    unittest.main()
```


#### Extracted Test Cases Code (Step 3)
```python
import unittest

class TestConvertNumberSystems(unittest.TestCase):

    def test_positive_integer(self):
        result = convert_number_systems(10)
        self.assertEqual(result["binary"], "0b1010")
        self.assertEqual(result["octal"], "0o12")
        self.assertEqual(result["hexadecimal"], "0xa")

    def test_zero(self):
        result = convert_number_systems(0)
        self.assertEqual(result["binary"], "0b0")
        self.assertEqual(result["octal"], "0o0")
        self.assertEqual(result["hexadecimal"], "0x0")

    def test_negative_integer(self):
        result = convert_number_systems(-10)
        self.assertEqual(result["binary"], "-0b1010")
        self.assertEqual(result["octal"], "-0o12")
        self.assertEqual(result["hexadecimal"], "-0xa")

    def test_large_integer(self):
        result = convert_number_systems(1024)
        self.assertEqual(result["binary"], "0b10000000000")
        self.assertEqual(result["hexadecimal"], "0x400")

    def test_invalid_input(self):
        with self.assertRaises(TypeError):
            convert_number_systems("string")
        with self.assertRaises(TypeError):
            convert_number_systems(None)

if __name__ == '__main__':
    unittest.main()
```
### Messages (Memory) after Step 3:
```json
[
  {
    "role": "user",
    "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."
  },
  {
    "role": "user",
    "content": "Write a basic Python function that Converts a decimal integer into its binary, octal, and hexadecimal string representations.."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into binary, octal, and hexadecimal strings.\n    Note: Python's built-in functions include prefixes (0b, 0o, 0x).\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }\n\n# Example usage:\nresult = convert_number_systems(255)\nprint(result)"
  },
  {
    "role": "user",
    "content": "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
  },
  {
    "role": "model",
    "content": "def convert_number_systems(n):\n    \"\"\"\n    Converts a decimal integer into its binary, octal, and hexadecimal string representations.\n\n    This function utilizes Python's built-in conversion functions. Note that the \n    returned strings include the standard Python prefixes: '0b' for binary, \n    '0o' for octal, and '0x' for hexadecimal.\n\n    Parameters:\n    n (int): The decimal integer to be converted.\n\n    Returns:\n    dict: A dictionary containing the original integer and its string \n          representations with the following keys:\n          - 'decimal': The original integer.\n          - 'binary': The binary string representation.\n          - 'octal': The octal string representation.\n          - 'hexadecimal': The hexadecimal string representation.\n\n    Example:\n    >>> convert_number_systems(255)\n    {'decimal': 255, 'binary': '0b11111111', 'octal': '0o377', 'hexadecimal': '0xff'}\n\n    Edge Cases:\n    - If n is 0, returns {'decimal': 0, 'binary': '0b0', 'octal': '0o0', 'hexadecimal': '0x0'}.\n    - If n is negative, the prefixes are preceded by a minus sign \n      (e.g., -0b101).\n    - If input is not an integer, a TypeError will be raised by the \n      built-in conversion functions.\n    \"\"\"\n    return {\n        \"decimal\": n,\n        \"binary\": bin(n),\n        \"octal\": oct(n),\n        \"hexadecimal\": hex(n)\n    }"
  },
  {
    "role": "user",
    "content": "Finally, add unittest test cases for this function. Tests should cover basic functionality, edge cases, and various input scenarios. Output the test cases code in a separate ```python code block```."
  },
  {
    "role": "model",
    "content": "import unittest\n\nclass TestConvertNumberSystems(unittest.TestCase):\n\n    def test_positive_integer(self):\n        result = convert_number_systems(10)\n        self.assertEqual(result[\"binary\"], \"0b1010\")\n        self.assertEqual(result[\"octal\"], \"0o12\")\n        self.assertEqual(result[\"hexadecimal\"], \"0xa\")\n\n    def test_zero(self):\n        result = convert_number_systems(0)\n        self.assertEqual(result[\"binary\"], \"0b0\")\n        self.assertEqual(result[\"octal\"], \"0o0\")\n        self.assertEqual(result[\"hexadecimal\"], \"0x0\")\n\n    def test_negative_integer(self):\n        result = convert_number_systems(-10)\n        self.assertEqual(result[\"binary\"], \"-0b1010\")\n        self.assertEqual(result[\"octal\"], \"-0o12\")\n        self.assertEqual(result[\"hexadecimal\"], \"-0xa\")\n\n    def test_large_integer(self):\n        result = convert_number_systems(1024)\n        self.assertEqual(result[\"binary\"], \"0b10000000000\")\n        self.assertEqual(result[\"hexadecimal\"], \"0x400\")\n\n    def test_invalid_input(self):\n        with self.assertRaises(TypeError):\n            convert_number_systems(\"string\")\n        with self.assertRaises(TypeError):\n            convert_number_systems(None)\n\nif __name__ == '__main__':\n    unittest.main()"
  }
]
```

## Final code has been saved to: converts_a_decimal_integer_int.py

---
### Quasi-Agent Development Process Complete
